# Agent-to-Agent Protocol (A2A) | Agent Protocols

In [1]:
import uuid
from typing import Callable, Dict, List

In [2]:
# --- A2A Server: serves Agent Card + handles tasks ---
class A2AServer:
    def __init__(self, name: str, skills: List[str], handler: Callable[[str, str], str]):
        self.agent_card = {
            "name": name, "description": f"{name} agent",
            "url": "http://localhost:9000",
            "skills": [{"id": s, "name": s} for s in skills],
            "version": "0.1.0",
        }
        self._handler = handler
        self._tasks: Dict[str, dict] = {}

    def handle_request(self, method: str, path: str, body: dict = None) -> dict:
        """Simulated HTTP endpoint dispatcher."""
        if method == "GET" and path == "/.well-known/agent.json":
            return {"status": 200, "body": self.agent_card}
        if method == "POST" and path == "/a2a" and body:
            params = body["params"]
            task_id = params.get("id", str(uuid.uuid4()))
            skill = params.get("skill", "default")
            text = params["message"]["parts"][0]["text"]
            self._tasks[task_id] = {"status": "working"}
            result = self._handler(skill, text)
            self._tasks[task_id] = {"status": "completed", "artifact": result}
            return {"status": 200, "body": {"jsonrpc": "2.0", "result": {
                "id": task_id, "status": "completed",
                "artifacts": [{"type": "text", "text": result}],
            }}}
        return {"status": 404, "body": {"error": "not found"}}

# --- A2A Client: discovers server capabilities, sends tasks ---
class A2AClient:
    def __init__(self, server: A2AServer):
        self._server = server  # simulated network connection

    def discover(self) -> dict:
        resp = self._server.handle_request("GET", "/.well-known/agent.json")
        return resp["body"]

    def send_task(self, skill: str, message: str) -> dict:
        payload = {
            "jsonrpc": "2.0", "method": "tasks/send",
            "params": {
                "id": f"task-{uuid.uuid4().hex[:8]}", "skill": skill,
                "message": {"role": "user", "parts": [{"type": "text", "text": message}]},
            },
        }
        return self._server.handle_request("POST", "/a2a", payload)["body"]

In [3]:
# --- Server agent: summarizes text ---
def summarizer(skill: str, text: str) -> str:
    words = text.split()
    return f"[{skill}] {' '.join(words[:10])}..." if len(words) > 10 else f"[{skill}] {text}"

server = A2AServer("SummaryBot", skills=["summarize", "translate"], handler=summarizer)
client = A2AClient(server)

# Step 1: Discover capabilities via Agent Card
card = client.discover()
skills = [s["id"] for s in card["skills"]]
print(f"Discovered: {card['name']} | Skills: {skills}")

Discovered: SummaryBot | Skills: ['summarize', 'translate']


In [4]:
# Step 2: Submit a task
result = client.send_task("summarize", "The quarterly earnings report shows revenue growth "
    "of 15% driven by strong performance in cloud services and AI products across all regions")
task = result["result"]
print(f"Task {task['id']}: status={task['status']}")
print(f"Artifact: {task['artifacts'][0]['text']}")

Task task-600c891d: status=completed
Artifact: [summarize] The quarterly earnings report shows revenue growth of 15% driven...


In [5]:
# Step 3: Submit another task with a different skill
result2 = client.send_task("translate", "Hello world")
print(f"Task {result2['result']['id']}: {result2['result']['artifacts'][0]['text']}")

Task task-1e4ca3e3: [translate] Hello world
